# 🍎 Responses API with AIProjectClient 🍏

In this notebook, we'll demonstrate how to generate conversational answers using the **Microsoft Foundry** SDK. We'll use the **`azure-ai-projects`** package and the Foundry project's **OpenAI-compatible client** (via `get_openai_client()`) to:

1. **Initialize** an `AIProjectClient`.
2. **Obtain** the project's OpenAI client to do direct LLM calls.
3. **Use** a **prompt template** to add system context.
4. **Send** user prompts in a health & fitness theme through the Responses API.

## 🏋️ Health-Fitness Disclaimer
> **This example is for demonstration only and does not provide real medical advice.** Always consult a professional for health or medical-related questions.

### Prerequisites
Before starting this notebook, please ensure you have completed all prerequisites listed in the root [README.md](../../README.md#-prerequisites).

Let's get started! 🎉

<img src="./seq-diagrams/1-chat.png" width="30%"/>


## 1. Initial Setup
Load environment variables, create an endpoint-based `AIProjectClient`, and get its **OpenAI client** (`get_openai_client()`) for Responses API calls. You'll also define a **prompt template** to show how you might structure system instructions.


In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

# Model deployment name (from Foundry > Models + endpoints)
model_deployment = os.environ.get("MODEL_DEPLOYMENT_NAME")

try:
    # Endpoint-based client (New Foundry) + its OpenAI client for Responses API
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ Successfully created AIProjectClient + OpenAI client")
except Exception as e:
    print("❌ Error initializing client:", e)

### Prompt Template
We'll define a quick **system** message that sets the context as a friendly, disclaimer-providing fitness assistant.

```txt
SYSTEM PROMPT (template):
You are FitChat GPT, a helpful fitness assistant.
Always remind users: I'm not a medical professional.
Be friendly, provide general advice.
...
```

We'll then pass user content as a **user** message.


In [ ]:
# We'll define a function that runs Responses API calls with system instructions and user input
def chat_with_fitness_assistant(user_input: str):
    """Use Responses API to get a response from our LLM, with system instructions."""
    # Our instruction template
    system_text = (
        "You are FitChat GPT, a friendly fitness assistant.\n"
        "Always remind users: I'm not a medical professional.\n"
        "Answer with empathy and disclaimers."
    )

    # Call Responses API via the OpenAI client (instructions + user input)
    response = openai_client.responses.create(
        model=model_deployment,
        instructions=system_text,
        input=user_input,
    )

    return response.output_text

print("Defined a helper function to use the Responses API.")

## 2. Try Responses API 🎉
We'll call the function with a user question about health or fitness and see the result. Feel free to modify the question or run multiple times!


In [ ]:
user_question = "How can I start a beginner workout routine at home?"
reply = chat_with_fitness_assistant(user_question)
print("🗣️ User:", user_question)
print("🤖 Assistant:", reply)

## 3. Another Example: Prompt Template with Fill-Ins 📝
We can go a bit further and add placeholders in the system message. For instance, imagine we have a **userName** or **goal**. We'll show a minimal example.


In [ ]:
def chat_with_template(user_input: str, user_name: str, goal: str):
    # Construct instruction template with placeholders
    system_template = (
        "You are FitChat GPT, an AI personal trainer for {name}.\n"
        "Your user wants to achieve: {goal}.\n"
        "Remind them you're not a medical professional. Offer friendly advice."
    )

    # Fill in placeholders
    system_prompt = system_template.format(name=user_name, goal=goal)

    response = openai_client.responses.create(
        model=model_deployment,
        instructions=system_prompt,
        input=user_input,
    )

    return response.output_text

# Let's try it out
templated_user_input = "What kind of home exercise do you recommend for a busy schedule?"
assistant_reply = chat_with_template(
    templated_user_input,
    user_name="Jordan",
    goal="increase muscle tone and endurance"
)
print("🗣️ User:", templated_user_input)
print("🤖 Assistant:", assistant_reply)

## 🎉 Congratulations!
You've successfully used the **Responses API** with Azure AI Foundry's `AIProjectClient` and its OpenAI-compatible client (`get_openai_client()`). You've also seen how to incorporate **prompt templates** to tailor your system instructions.

#### Head to [2-embeddings.ipynb](2-embeddings.ipynb) for the next part of the workshop! 🎯